In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-04-17 15:44:20 +05:30)


In [2]:
import tensorflow as tf
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import random

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import layers, models, regularizers

from sklearn.utils.class_weight import compute_class_weight

time: 8.52 s (started: 2026-04-17 15:44:20 +05:30)


In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True
time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [6]:
CLASSES = ["NORMAL", "PNEUMONIA"]

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [7]:
bad_files = (
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpeg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.png"))
)

deleted = 0

for f in bad_files:
    try:
        f.unlink()
        deleted += 1
    except Exception as e:
        print(f"Error deleting {f}: {e}")

print(f"Deleted {deleted} augmented files")

Deleted 0 augmented files
time: 31 ms (started: 2026-04-17 15:44:28 +05:30)


In [8]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


Critical nuance — the rescale here does NOT apply during training. The custom generator (make_balanced_fold_gen) calls datagen.random_transform(), which applies only the geometric transforms. rescale is only applied by Keras's own flow_* methods — not by random_transform. The rescaling in this notebook is instead done manually (img / 255.0) inside the generator. So rescale=1/255 inside train_datagen is redundant but harmless — it simply has no effect.

In [9]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [10]:
def make_balanced_fold_gen(paths, labels, batch_size=32, datagen=None):
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    while True:
        batch_paths, batch_labels = [], []

        for _ in range(batch_size // 2):
            p = np.random.choice(pos_idx)
            n = np.random.choice(neg_idx)

            batch_paths += [paths[p], paths[n]]
            batch_labels += [1, 0]

        combined = list(zip(batch_paths, batch_labels))
        random.shuffle(combined)
        batch_paths, batch_labels = zip(*combined)

        images = []
        for path in batch_paths:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0
        
            img = np.expand_dims(img, axis=-1)  
        
            if datagen is not None:
                img = datagen.random_transform(img)  
        
            images.append(img)

        yield np.array(images), np.array(batch_labels)

time: 15 ms (started: 2026-04-17 15:44:28 +05:30)


In [11]:
label_to_classname = {0: "NORMAL", 1: "PNEUMONIA"}

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [12]:
def make_val_generator(val_paths, val_labels):

    val_df = pd.DataFrame({
        "filename": val_paths,
        "class": [label_to_classname[l] for l in val_labels]
    })

    return val_test_datagen.flow_from_dataframe(
        val_df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        color_mode=COLOR_MODE,
        shuffle=False
    )

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


In [13]:
def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        layers.Conv2D(32, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        layers.Conv2D(64, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )

    return m

time: 0 ns (started: 2026-04-17 15:44:28 +05:30)


Execution order reminder: The below cell must run after the deletion cell. If you re-run the notebook top-to-bottom, this is fine. If you re-run only this cell after training, deleted aug_* files won't be in all_paths either. Just be careful not to run this cell before the deletion cell.

In [14]:
VALID_EXTS = {".jpeg", ".jpg", ".png"}
 
all_paths = []
all_labels = []

temp_gen = train_datagen.flow_from_directory( #dummy generator
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode="binary",
    shuffle=False
)

class_map = temp_gen.class_indices  
 
for class_name, label in class_map.items():
    class_dir = TRAIN_DIR / class_name
 
    for img_name in os.listdir(class_dir):
        if Path(img_name).suffix.lower() not in VALID_EXTS:
            continue
        all_paths.append(str(class_dir / img_name))
        all_labels.append(label)
 
all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
 
print("Total samples:", len(all_paths))
print("Class distribution:", np.bincount(all_labels))

Found 4679 images belonging to 2 classes.
Total samples: 4679
Class distribution: [1199 3480]
time: 531 ms (started: 2026-04-17 15:44:28 +05:30)


VALID_EXTS bug confirmed fixed. Path(img_name).suffix correctly extracts the extension from a bare filename (e.g., "image.jpeg" → ".jpeg"). This works regardless of whether img_name is a full path or just a filename. The previous concern about this was unfounded — Path.suffix works on filename strings directly.

In [15]:
def custom_stratified_kfold_split(paths, labels, n_splits=3, shuffle=True, random_state=42):
    """
    Custom implementation of StratifiedKFold.split().
    Yields (train_idx, val_idx) for each fold, preserving class proportions.
    """
    rng = np.random.RandomState(random_state)   # legacy RandomState

    # Group indices by class label
    unique_classes = np.unique(labels)
    class_indices = {cls: np.where(labels == cls)[0] for cls in unique_classes}
    
    # Optionally shuffle within each class
    if shuffle:
        for cls in unique_classes:
            rng.shuffle(class_indices[cls])
    
    # Split each class's indices into n_splits roughly equal folds
    class_folds = {}
    for cls in unique_classes:
        class_folds[cls] = np.array_split(class_indices[cls], n_splits)
    
    # For each fold, val = that fold's chunk per class, train = the rest
    for fold_idx in range(n_splits):
        val_indices = []
        train_indices = []
        
        for cls in unique_classes:
            val_indices.append(class_folds[cls][fold_idx])
            train_indices.extend(
                class_folds[cls][i] for i in range(n_splits) if i != fold_idx
            )
        
        val_idx   = np.concatenate(val_indices)
        train_idx = np.concatenate(train_indices)
        
        # Shuffle the final index arrays so classes aren't in blocks
        if shuffle:
            rng.shuffle(val_idx)
            rng.shuffle(train_idx)
        
        yield train_idx, val_idx

time: 15 ms (started: 2026-04-17 15:44:29 +05:30)


In [16]:
for fold, (train_idx, val_idx) in enumerate(
    custom_stratified_kfold_split(all_paths, all_labels, n_splits=3, shuffle=True, random_state=42), 1
):
    print(f"\n------------ Fold {fold} ------------")

    train_paths_fold  = all_paths[train_idx]
    val_paths_fold    = all_paths[val_idx]
    train_labels_fold = all_labels[train_idx]
    val_labels_fold   = all_labels[val_idx]

    train_gen = make_balanced_fold_gen(
        train_paths_fold,
        train_labels_fold,
        batch_size=BATCH_SIZE,
        datagen=train_datagen
    )

    val_gen = make_val_generator(val_paths_fold, val_labels_fold)
    model   = build_model()

    steps_per_epoch = len(train_paths_fold) // BATCH_SIZE

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
        ModelCheckpoint(
            filepath=f"best_model_kfoldc_{fold}.h5",
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen,
        epochs=15,
        callbacks=callbacks
    )


------------ Fold 1 ------------
Found 1560 validated image filenames belonging to 2 classes.
Epoch 1/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 947ms/step - accuracy: 0.5708 - loss: 2.5467 - precision: 0.5695 - recall: 0.5776
Epoch 1: val_accuracy improved from None to 0.74359, saving model to best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.6343 - loss: 1.1624 - precision: 0.6277 - recall: 0.6604 - val_accuracy: 0.7436 - val_loss: 0.6280 - val_precision: 0.7436 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 924ms/step - accuracy: 0.7147 - loss: 0.5218 - precision: 0.6776 - recall: 0.8216
Epoch 2: val_accuracy did not improve from 0.74359
97/97 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.7445 - loss: 0.4740 - precision: 0.7093 - recall: 0.8286 - val_accuracy: 0.7436 - val_loss: 0.5595 - val_precision: 0.7436 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 914ms/step - accuracy: 0.7967 - loss: 0.3865 - precision: 0.7613 - recall: 0.8657
Epoch 3: val_accuracy did not improve from 0.74359
97/97 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.8022 - loss: 0.3851 - precision: 0.7775 - recall: 0.8466 - val_accuracy: 0.7436 - val_loss: 0.6015 - val_precision: 0.7436 - val_recall: 1.0000 - 

97/97 ━━━━━━━━━━━━━━━━━━━━ 96s 988ms/step - accuracy: 0.8756 - loss: 0.3063 - precision: 0.9054 - recall: 0.8389 - val_accuracy: 0.7545 - val_loss: 0.3856 - val_precision: 0.7518 - val_recall: 1.0000 - learning_rate: 3.0000e-05
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 764ms/step - accuracy: 0.8720 - loss: 0.3175 - precision: 0.9016 - recall: 0.8352
Epoch 6: val_accuracy improved from 0.75449 to 0.92115, saving model to best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 89s 919ms/step - accuracy: 0.8779 - loss: 0.3057 - precision: 0.9122 - recall: 0.8363 - val_accuracy: 0.9212 - val_loss: 0.2138 - val_precision: 0.9092 - val_recall: 0.9931 - learning_rate: 3.0000e-05
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 755ms/step - accuracy: 0.8708 - loss: 0.3014 - precision: 0.9088 - recall: 0.8246
Epoch 7: val_accuracy improved from 0.92115 to 0.94167, saving model to best_model_kfoldc_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 88s 908ms/step - accuracy: 0.8753 - loss: 0.3034 - precision: 0.9082 - recall: 0.8351 - val_accuracy: 0.9417 - val_loss: 0.1837 - val_precision: 0.9890 - val_recall: 0.9319 - learning_rate: 3.0000e-05
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 760ms/step - accuracy: 0.8879 - loss: 0.2881 - precision: 0.9210 - recall: 0.8486
Epoch 8: val_accuracy did not improve from 0.94167
97/97 ━━━━━━━━━━━━━━━━━━━━ 88s 908ms/step - accuracy: 0.8860 - loss: 0.2890 - precision: 0.9218 - recall: 0.8434 - val_accuracy: 0.9115 - val_loss: 0.2161 - val_precision: 0.9932 - val_recall: 0.8871 - learning_rate: 3.0000e-05
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 772ms/step - accuracy: 0.8922 - loss: 0.2801 - precision: 0.9166 - recall: 0.8630
Epoch 9: val_accuracy did not improve from 0.94167
97/97 ━━━━━━━━━━━━━━━━━━━━ 89s 920ms/step - accuracy: 0.8898 - loss: 0.2869 - precision: 0.9167 - recall: 0.8576 - val_accuracy: 0.8974 - val_loss: 0.2295 - val_precision: 0.9941 - val_recall: 0.8

...

97/97 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.7162 - loss: 1.5002 - precision_2: 0.7172 - recall_2: 0.7139 - val_accuracy: 0.7441 - val_loss: 0.5541 - val_precision_2: 0.7441 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 930ms/step - accuracy: 0.8456 - loss: 0.3671 - precision_2: 0.8397 - recall_2: 0.8544
Epoch 2: val_accuracy did not improve from 0.74407
97/97 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8528 - loss: 0.3537 - precision_2: 0.8494 - recall_2: 0.8576 - val_accuracy: 0.7441 - val_loss: 0.6061 - val_precision_2: 0.7441 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 979ms/step - accuracy: 0.8812 - loss: 0.2955 - precision_2: 0.8810 - recall_2: 0.8815
Epoch 3: val_accuracy did not improve from 0.74407
97/97 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.8795 - loss: 0.2968 - precision_2: 0.8800 - recall_2: 0.8789 - val_accuracy: 0.7441 - val_loss: 0.6912 - val_precision_2: 

97/97 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.9124 - loss: 0.2442 - precision_2: 0.9145 - recall_2: 0.9098 - val_accuracy: 0.7447 - val_loss: 0.5721 - val_precision_2: 0.7445 - val_recall_2: 1.0000 - learning_rate: 3.0000e-05
Epoch 5/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 859ms/step - accuracy: 0.9091 - loss: 0.2332 - precision_2: 0.9126 - recall_2: 0.9050
Epoch 5: val_accuracy improved from 0.74471 to 0.81783, saving model to best_model_kfoldc_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.9104 - loss: 0.2304 - precision_2: 0.9136 - recall_2: 0.9066 - val_accuracy: 0.8178 - val_loss: 0.3367 - val_precision_2: 0.8046 - val_recall_2: 0.9974 - learning_rate: 3.0000e-05
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 814ms/step - accuracy: 0.9125 - loss: 0.2458 - precision_2: 0.9208 - recall_2: 0.9027
Epoch 6: val_accuracy improved from 0.81783 to 0.89609, saving model to best_model_kfoldc_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 971ms/step - accuracy: 0.9146 - loss: 0.2352 - precision_2: 0.9209 - recall_2: 0.9072 - val_accuracy: 0.8961 - val_loss: 0.2297 - val_precision_2: 0.8844 - val_recall_2: 0.9897 - learning_rate: 3.0000e-05
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 816ms/step - accuracy: 0.9094 - loss: 0.2231 - precision_2: 0.9100 - recall_2: 0.9089
Epoch 7: val_accuracy improved from 0.89609 to 0.94355, saving model to best_model_kfoldc_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 972ms/step - accuracy: 0.9127 - loss: 0.2324 - precision_2: 0.9156 - recall_2: 0.9091 - val_accuracy: 0.9436 - val_loss: 0.1671 - val_precision_2: 0.9820 - val_recall_2: 0.9414 - learning_rate: 3.0000e-05
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 792ms/step - accuracy: 0.9211 - loss: 0.2122 - precision_2: 0.9327 - recall_2: 0.9078
Epoch 8: val_accuracy did not improve from 0.94355
97/97 ━━━━━━━━━━━━━━━━━━━━ 91s 942ms/step - accuracy: 0.9198 - loss: 0.2165 - precision_2: 0.9267 - recall_2: 0.9117 - val_accuracy: 0.9070 - val_loss: 0.2689 - val_precision_2: 0.9894 - val_recall_2: 0.8845 - learning_rate: 3.0000e-05
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - accuracy: 0.9149 - loss: 0.2269 - precision_2: 0.9278 - recall_2: 0.9005
Epoch 9: val_accuracy did not improve from 0.94355
97/97 ━━━━━━━━━━━━━━━━━━━━ 90s 931ms/step - accuracy: 0.9159 - loss: 0.2258 - precision_2: 0.9222 - recall_2: 0.9085 - val_accuracy: 0.9051 - val_loss: 0.2731 - val_precisi

In [17]:
print("end")

end
time: 0 ns (started: 2026-04-17 17:26:03 +05:30)
